# Day 10 — Finalization and Packaging

Day 10 performs finalization and packaging only. No new model selection or KDDTest+-based tuning is performed.

Days 1–9 are complete and are **not** modified. The locked model remains Random Forest, 500 trees, `class_weight=None`, no SMOTE, no threshold tuning, selected on KDDTrain+ validation only.

## 1. Scope and Final Model

| Item | Locked value |
|---|---|
| Model | Random Forest |
| `n_estimators` | 500 |
| `random_state` | 42 |
| `class_weight` | None |
| SMOTE | No |
| Threshold tuning | No |
| Selection | KDDTrain+ validation only |
| Validation Macro F1 | 0.951031 |
| Validation Accuracy | 0.998809 |
| KDDTest+ Accuracy | 0.7447 (evaluation only; not used here) |
| KDDTest+ Macro F1 | 0.5061 (evaluation only; not used here) |

This notebook packages that configuration as a `sklearn.pipeline.Pipeline` (preprocessor + classifier), verifies it on the **validation** split only, and records metadata. **KDDTest+ is not scored.**

## 2. Environment and Repository Audit

In [19]:
from pathlib import Path
import json
import platform
import subprocess
import sys

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

here = Path.cwd().resolve()
if (here / "notebooks" / "05_random_forest.ipynb").exists():
    ROOT = here
elif (here / "05_random_forest.ipynb").exists():
    ROOT = here.parent
else:
    ROOT = Path("..").resolve()

NB_DIR = ROOT / "notebooks"
MODELS_DIR = ROOT / "models"
DATA_TRAIN = ROOT / "data" / "raw" / "KDDTrain+.txt"

print("Day 10 performs finalization and packaging only. No new model selection or KDDTest+-based tuning is performed.")
print()
print("Python version:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Repository root:", ROOT)

def git(*args):
    r = subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True)
    return r.stdout.strip()

print("Git branch:", git("rev-parse", "--abbrev-ref", "HEAD"))
print("Git status:")
print(git("status", "-sb"))
print()

expected_nbs = [
    "01_eda.ipynb",
    "02_baseline_model.ipynb",
    "04_smote_experiments.ipynb",
    "05_random_forest.ipynb",
    "06_model_analysis.ipynb",
    "07_final_model_comparison.ipynb",
    "08_final_evaluation.ipynb",
    "09_reproducibility_audit.ipynb",
    "10_finalization_and_packaging.ipynb",
]
inv = []
for name in expected_nbs:
    p = NB_DIR / name
    inv.append({"Notebook": f"notebooks/{name}", "Exists": p.exists(), "Size (bytes)": p.stat().st_size if p.exists() else None})
display(pd.DataFrame(inv))
print("03_*.ipynb is intentionally absent.")
print()
print("models/ exists:", MODELS_DIR.exists())
print("requirements.txt exists:", (ROOT / "requirements.txt").exists())
print(".gitignore exists:", (ROOT / ".gitignore").exists())
readme_hits = list(ROOT.glob("README*"))
print("README present:", bool(readme_hits), readme_hits)


Day 10 performs finalization and packaging only. No new model selection or KDDTest+-based tuning is performed.

Python version: 3.10.11
Platform: Windows-10-10.0.26200-SP0
Repository root: D:\IIITB\network-traffic-anomaly-detection
Git branch: main
Git status:
## main...origin/main
?? models/
?? notebooks/10_finalization_and_packaging.ipynb



,Notebook,Exists,Size (bytes)
0,notebooks/01_eda.ipynb,True,125488
1,notebooks/02_baseline_model.ipynb,True,776308
2,notebooks/04_smote_experiments.ipynb,True,255434
3,notebooks/05_random_forest.ipynb,True,92615
4,notebooks/06_model_analysis.ipynb,True,296813
5,notebooks/07_final_model_comparison.ipynb,True,18717
6,notebooks/08_final_evaluation.ipynb,True,23344
7,notebooks/09_reproducibility_audit.ipynb,True,28077
8,notebooks/10_finalization_and_packaging.ipynb,True,30288


03_*.ipynb is intentionally absent.

models/ exists: True
requirements.txt exists: True
.gitignore exists: True
README present: False []


## 3. Day 5 Locked Model Audit

Items below are checked against `notebooks/05_random_forest.ipynb` source and saved outputs, not assumed.

In [20]:
def notebook_blob(path):
    nb = json.loads(path.read_text(encoding="utf-8"))
    parts = []
    for cell in nb.get("cells", []):
        parts.append("".join(cell.get("source", [])))
        for out in cell.get("outputs", []) or []:
            if out.get("output_type") == "stream":
                t = out.get("text", [])
                parts.append("".join(t) if isinstance(t, list) else str(t))
    return "\n".join(parts)

d5_path = NB_DIR / "05_random_forest.ipynb"
d5 = notebook_blob(d5_path) if d5_path.exists() else ""

audit_items = [
    ("RandomForestClassifier", "RandomForestClassifier" in d5),
    ("n_estimators = 500", "n_estimators=500" in d5 or "n_estimators: 500" in d5),
    ("random_state = 42", "random_state=42" in d5),
    ("class_weight = None", "class_weight=None" in d5 or "class_weight: None" in d5),
    ("no SMOTE", "no SMOTE" in d5 or "No SMOTE" in d5),
    ("no threshold tuning", "threshold tuning" in d5.lower()),
    ("OneHotEncoder(handle_unknown=ignore)", 'handle_unknown="ignore"' in d5 or "handle_unknown='ignore'" in d5),
    ("StandardScaler on numeric features", "StandardScaler" in d5),
    ("ColumnTransformer preprocessor", "ColumnTransformer" in d5),
    ("80/20 stratified split random_state=42", "test_size=0.20" in d5 and "stratify" in d5 and "random_state=42" in d5),
    ("KDDTest+ held out during selection", "KDDTest+ has not been used for model selection" in d5 or "validation only" in d5),
    ("Selected using validation Macro F1", "0.951031" in d5),
]
day10_day5_audit = pd.DataFrame(
    [{"Item": k, "Found in Day 5 notebook": v} for k, v in audit_items]
)
display(day10_day5_audit)
print("Source file:", d5_path)


,Item,Found in Day 5 notebook
0,RandomForestClassifier,True
1,n_estimators = 500,True
2,random_state = 42,True
3,class_weight = None,True
4,no SMOTE,True
5,no threshold tuning,True
6,OneHotEncoder(handle_unknown=ignore),True
7,StandardScaler on numeric features,True
8,ColumnTransformer preprocessor,True
9,80/20 stratified split random_state=42,True


Source file: D:\IIITB\network-traffic-anomaly-detection\notebooks\05_random_forest.ipynb


## 4. Deployable Pipeline Construction

Same Day 5 schema, 80/20 stratified KDDTrain+ split (`random_state=42`), preprocessor (OHE ignore + StandardScaler), classifier as locked. Fit **only** on `X_train_fit` / `y_train_fit`. Validation and KDDTest+ are not used for fitting.

In [21]:
COLUMNS = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in",
    "num_compromised", "root_shell", "su_attempted", "num_root", "num_file_creations",
    "num_shells", "num_access_files", "num_outbound_cmds", "is_host_login",
    "is_guest_login", "count", "srv_count", "serror_rate", "srv_serror_rate",
    "rerror_rate", "srv_rerror_rate", "same_srv_rate", "diff_srv_rate",
    "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate", "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate", "dst_host_serror_rate", "dst_host_srv_serror_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate", "label", "difficulty",
]
attack_category_map = {
    "normal": "Normal", "back": "DoS", "land": "DoS", "neptune": "DoS", "pod": "DoS",
    "smurf": "DoS", "teardrop": "DoS", "apache2": "DoS", "udpstorm": "DoS",
    "processtable": "DoS", "mailbomb": "DoS", "worm": "DoS", "satan": "Probe",
    "ipsweep": "Probe", "nmap": "Probe", "portsweep": "Probe", "mscan": "Probe",
    "saint": "Probe", "guess_passwd": "R2L", "ftp_write": "R2L", "imap": "R2L",
    "phf": "R2L", "multihop": "R2L", "warezmaster": "R2L", "warezclient": "R2L",
    "spy": "R2L", "xlock": "R2L", "xsnoop": "R2L", "snmpguess": "R2L",
    "snmpgetattack": "R2L", "httptunnel": "R2L", "sendmail": "R2L", "named": "R2L",
    "buffer_overflow": "U2R", "loadmodule": "U2R", "rootkit": "U2R", "perl": "U2R",
    "sqlattack": "U2R", "xterm": "U2R", "ps": "U2R",
}
category_order = ["Normal", "DoS", "Probe", "R2L", "U2R"]
feature_columns = [c for c in COLUMNS if c not in ("label", "difficulty")]
categorical_features = ["protocol_type", "service", "flag"]
numerical_features = [c for c in feature_columns if c not in categorical_features]

if not DATA_TRAIN.exists():
    print("STOPPED. Missing training file:", DATA_TRAIN)
    print("Do not reconstruct from KDDTest+. Packaging cannot proceed without KDDTrain+.")
else:
    train_df = pd.read_csv(DATA_TRAIN, header=None, names=COLUMNS)
    y_train = train_df["label"].map(attack_category_map)
    X_train = train_df[feature_columns]
    X_train_fit, X_validation, y_train_fit, y_validation = train_test_split(
        X_train, y_train, test_size=0.20, stratify=y_train, random_state=42
    )
    print("X_train_fit:", X_train_fit.shape, "X_validation:", X_validation.shape)
    print("Fit partition only. X_test / y_test are not loaded and are not used.")

    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
            ("num", StandardScaler(), numerical_features),
        ]
    )
    classifier = RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        n_jobs=-1,
        class_weight=None,
    )
    rf_deploy_pipeline = Pipeline(
        steps=[("preprocessor", preprocessor), ("classifier", classifier)]
    )
    rf_deploy_pipeline.fit(X_train_fit, y_train_fit)
    print("Pipeline fitted on X_train_fit / y_train_fit only.")


X_train_fit: (100778, 41) X_validation: (25195, 41)
Fit partition only. X_test / y_test are not loaded and are not used.
Pipeline fitted on X_train_fit / y_train_fit only.


## 5. Model Artifact Serialization

In [22]:
if "rf_deploy_pipeline" not in globals():
    print("STOPPED. rf_deploy_pipeline is unavailable. No artifact written.")
else:
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    artifact_path = MODELS_DIR / "nsl_kdd_random_forest_500.joblib"
    joblib.dump(rf_deploy_pipeline, artifact_path)
    print("Saved:", artifact_path.resolve())
    print("File size (bytes):", artifact_path.stat().st_size)
    print("File size (MiB):", round(artifact_path.stat().st_size / (1024 * 1024), 2))
    print("artifact exists =", artifact_path.exists())
    print("Not saved inside .venv.")


Saved: D:\IIITB\network-traffic-anomaly-detection\models\nsl_kdd_random_forest_500.joblib
File size (bytes): 59053402
File size (MiB): 56.32
artifact exists = True
Not saved inside .venv.


## 6. Artifact Reload and Integrity Check

In [23]:
artifact_path = MODELS_DIR / "nsl_kdd_random_forest_500.joblib"
if not artifact_path.exists():
    print("STOPPED. Artifact missing:", artifact_path)
else:
    loaded_model = joblib.load(artifact_path)
    clf = loaded_model.named_steps["classifier"] if isinstance(loaded_model, Pipeline) else None
    integrity = pd.DataFrame(
        [
            {"Check": "Loads successfully", "Result": True},
            {"Check": "Type is Pipeline", "Result": isinstance(loaded_model, Pipeline)},
            {"Check": "Step preprocessor", "Result": "preprocessor" in getattr(loaded_model, "named_steps", {})},
            {"Check": "Step classifier", "Result": "classifier" in getattr(loaded_model, "named_steps", {})},
            {"Check": "Classifier is RandomForestClassifier", "Result": isinstance(clf, RandomForestClassifier)},
            {"Check": "n_estimators = 500", "Result": getattr(clf, "n_estimators", None) == 500},
            {"Check": "class_weight is None", "Result": getattr(clf, "class_weight", "missing") is None},
            {"Check": "random_state = 42", "Result": getattr(clf, "random_state", None) == 42},
        ]
    )
    display(integrity)


,Check,Result
0,Loads successfully,True
1,Type is Pipeline,True
2,Step preprocessor,True
3,Step classifier,True
4,Classifier is RandomForestClassifier,True
5,n_estimators = 500,True
6,class_weight is None,True
7,random_state = 42,True


## 7. Packaged Model Validation Smoke Test

Validation split only. This is **not** a new experiment and **not** a KDDTest+ evaluation.

In [24]:
expected_val = {
    "Accuracy": 0.998809,
    "Macro Precision": 0.974485,
    "Macro Recall": 0.931597,
    "Macro F1": 0.951031,
    "R2L F1": 0.982097,
    "U2R F1": 0.777778,
}

if "loaded_model" not in globals() or "X_validation" not in globals():
    print("STOPPED. Need loaded_model and X_validation. No KDDTest+ fallback.")
else:
    y_validation_loaded = loaded_model.predict(X_validation)
    print("Prediction count:", len(y_validation_loaded), "validation count:", len(y_validation))
    print("Counts match:", len(y_validation_loaded) == len(y_validation))
    print("Labels:", sorted(set(y_validation_loaded)))
    print("Labels subset of category_order:", set(y_validation_loaded).issubset(set(category_order)))

    acc = accuracy_score(y_validation, y_validation_loaded)
    mp = precision_score(y_validation, y_validation_loaded, average="macro", labels=category_order, zero_division=0)
    mr = recall_score(y_validation, y_validation_loaded, average="macro", labels=category_order, zero_division=0)
    mf = f1_score(y_validation, y_validation_loaded, average="macro", labels=category_order, zero_division=0)
    f1s = f1_score(y_validation, y_validation_loaded, average=None, labels=category_order, zero_division=0)
    per = dict(zip(category_order, f1s))

    day10_packaged_validation = pd.DataFrame(
        [
            {
                "Metric": k,
                "Expected (Day 5)": v,
                "Packaged artifact": {
                    "Accuracy": acc,
                    "Macro Precision": mp,
                    "Macro Recall": mr,
                    "Macro F1": mf,
                    "R2L F1": per["R2L"],
                    "U2R F1": per["U2R"],
                }[k],
            }
            for k, v in expected_val.items()
        ]
    )
    day10_packaged_validation["Abs difference"] = (
        day10_packaged_validation["Packaged artifact"] - day10_packaged_validation["Expected (Day 5)"]
    ).abs()
    display(day10_packaged_validation)
    print("KDDTest+ was not used. This smoke test is validation-only packaging verification.")


Prediction count: 25195 validation count: 25195
Counts match: True
Labels: ['DoS', 'Normal', 'Probe', 'R2L', 'U2R']
Labels subset of category_order: True


,Metric,Expected (Day 5),Packaged artifact,Abs difference
0,Accuracy,0.998809,0.998809,2.875571e-07
1,Macro Precision,0.974485,0.974485,3.004464e-07
2,Macro Recall,0.931597,0.931597,3.450778e-07
3,Macro F1,0.951031,0.951031,3.161929e-07
4,R2L F1,0.982097,0.982097,1.867008e-07
5,U2R F1,0.777778,0.777778,2.222222e-07


KDDTest+ was not used. This smoke test is validation-only packaging verification.


## 8. Model Metadata

In [25]:
metadata = {
    "model_name": "nsl_kdd_random_forest_500",
    "model_type": "sklearn.pipeline.Pipeline[ColumnTransformer + RandomForestClassifier]",
    "n_estimators": 500,
    "random_state": 42,
    "class_weight": None,
    "smote_used": False,
    "threshold_tuning": False,
    "selection_split": "KDDTrain+ stratified validation (test_size=0.20, random_state=42)",
    "training_partition": "X_train_fit / y_train_fit only",
    "validation_macro_f1": 0.951031,
    "validation_accuracy": 0.998809,
    "kddtest_note": "KDDTest+ was not used for model selection.",
    "artifact_filename": "nsl_kdd_random_forest_500.joblib",
    "python_version": sys.version.split()[0],
    "sklearn_version": sklearn.__version__,
    "joblib_version": joblib.__version__,
}
meta_path = MODELS_DIR / "nsl_kdd_random_forest_500_metadata.json"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
meta_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Wrote:", meta_path.resolve())
print(json.dumps(metadata, indent=2))
print()
print("KDDTest+ was not used for model selection.")


Wrote: D:\IIITB\network-traffic-anomaly-detection\models\nsl_kdd_random_forest_500_metadata.json
{
  "model_name": "nsl_kdd_random_forest_500",
  "model_type": "sklearn.pipeline.Pipeline[ColumnTransformer + RandomForestClassifier]",
  "n_estimators": 500,
  "random_state": 42,
  "class_weight": null,
  "smote_used": false,
  "threshold_tuning": false,
  "selection_split": "KDDTrain+ stratified validation (test_size=0.20, random_state=42)",
  "training_partition": "X_train_fit / y_train_fit only",
  "validation_macro_f1": 0.951031,
  "validation_accuracy": 0.998809,
  "kddtest_note": "KDDTest+ was not used for model selection.",
  "artifact_filename": "nsl_kdd_random_forest_500.joblib",
  "python_version": "3.10.11",
  "sklearn_version": "1.7.2",
  "joblib_version": "1.5.2"
}

KDDTest+ was not used for model selection.


## 9. Deployment Input Contract

A future backend must pass a table (or equivalent records) with the **same feature names and types** the Day 5 pipeline was fitted on. Do not drop, rename, or reorder columns unless the backend maps them back to this schema before `predict`.

**Output classes (five-class `attack_category`):** Normal, DoS, Probe, R2L, U2R.

In [26]:
print("Total feature count:", len(feature_columns))
print("Categorical:", categorical_features)
print("Numerical count:", len(numerical_features))
print("Expected column order:")
print(feature_columns)
print()
print("Output classes:", category_order)
contract = {
    "n_features": len(feature_columns),
    "categorical_features": categorical_features,
    "numerical_features": numerical_features,
    "feature_columns_in_order": feature_columns,
    "output_classes": category_order,
}
(MODELS_DIR / "nsl_kdd_random_forest_500_input_contract.json").write_text(
    json.dumps(contract, indent=2), encoding="utf-8"
)
print("Also wrote models/nsl_kdd_random_forest_500_input_contract.json")


Total feature count: 41
Categorical: ['protocol_type', 'service', 'flag']
Numerical count: 38
Expected column order:
['duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate']

Output classes: ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']
Also wrote models/nsl_kdd_random_forest_500_input_contract.json


## 10. Environment and Dependency Record

No packages are upgraded or added here. `requirements.txt` is inspected only.

In [27]:
import importlib.metadata as im

def ver(name):
    try:
        return im.version(name)
    except im.PackageNotFoundError:
        return "not installed"

print("python", sys.version.split()[0])
print("numpy", np.__version__)
print("pandas", pd.__version__)
print("scikit-learn", sklearn.__version__)
print("joblib", joblib.__version__)
print("matplotlib", ver("matplotlib"))
print("xgboost", ver("xgboost"))
print()
req = ROOT / "requirements.txt"
if req.exists():
    print("requirements.txt:")
    print(req.read_text(encoding="utf-8"))
else:
    print("requirements.txt is absent. Not creating extra unused packages in this cell.")


python 3.10.11
numpy 2.2.6
pandas 2.3.3
scikit-learn 1.7.2
joblib 1.5.2
matplotlib 3.10.7
xgboost 3.2.0

requirements.txt:
pandas==2.3.3
numpy==2.2.6
scikit-learn==1.7.2
xgboost==3.2.0
matplotlib==3.10.7
seaborn==0.13.2
joblib==1.5.2
ipykernel==6.30.1



## 11. Git / Repository Safety Check

No commit and no push from this notebook.

In [28]:
gi = (ROOT / ".gitignore").read_text(encoding="utf-8") if (ROOT / ".gitignore").exists() else ""
checks = {
    ".venv ignored": ".venv" in gi,
    "__pycache__/ ignored": "__pycache__/" in gi,
    ".ipynb_checkpoints ignored": ".ipynb_checkpoints" in gi,
    "data/raw/* ignored": "data/raw/*" in gi,
}
for k, v in checks.items():
    print(f"{k}: {v}")
print("models/ ignored?:", any(line.strip() == "models/" or line.strip() == "models/*" for line in gi.splitlines()))
print()
art = MODELS_DIR / "nsl_kdd_random_forest_500.joblib"
if art.exists():
    size_mb = art.stat().st_size / (1024 * 1024)
    print(f"Model artifact size: {size_mb:.2f} MiB")
    if size_mb > 50:
        print("Recommendation: artifact is large; consider Git LFS or external artifact storage rather than a plain Git blob.")
    else:
        print("Recommendation: size is reasonable for a normal Git commit if the team accepts binary model files.")
print()
print("Git status (no commit):")
print(git("status", "-sb"))


.venv ignored: True
__pycache__/ ignored: True
.ipynb_checkpoints ignored: True
data/raw/* ignored: True
models/ ignored?: False

Model artifact size: 56.32 MiB
Recommendation: artifact is large; consider Git LFS or external artifact storage rather than a plain Git blob.

Git status (no commit):
## main...origin/main
?? models/
?? notebooks/10_finalization_and_packaging.ipynb


## 12. Final Project Summary

1. **Problem.** Five-class network traffic anomaly detection: Normal, DoS, Probe, R2L, U2R.
2. **Dataset.** NSL-KDD (`KDDTrain+` / `KDDTest+`), 41 features plus label and difficulty.
3. **Preprocessing.** One-hot encode `protocol_type`, `service`, `flag` (`handle_unknown="ignore"`); `StandardScaler` on remaining numeric columns; fitted on the training partition only.
4. **Baseline models.** Days 2–3: Logistic Regression and other baselines, including class weighting, on a held-out protocol that later used a KDDTrain+ validation split.
5. **SMOTE experiments.** Day 4 compared LR without SMOTE vs LR with SMOTE on validation, then scored KDDTest+ once after that comparison.
6. **Random Forest selection.** Day 5 compared 100/300/500 trees on validation only; **500 trees** won validation Macro F1 and was locked.
7. **Validation performance.** Accuracy 0.998809, Macro F1 0.951031, R2L F1 0.982097, U2R F1 0.777778.
8. **KDDTest+ performance.** Accuracy 0.7447, Macro F1 0.5061 (evaluation after lock).
9. **Generalization gap.** Macro F1 0.951031 → 0.5061 (absolute drop 0.4449, about 46.78% relative).
10. **Distribution shift.** R2L prevalence 0.7898% → 12.7972%; U2R 0.0397% → 0.2972%; largest overall numerical |SMD| 0.5060. This is **consistent with** dataset/domain shift and **may help explain** the gap. It does **not** prove that any individual feature caused the errors.
11. **Final locked model.** Random Forest, 500 trees, `random_state=42`, `class_weight=None`, no SMOTE, no threshold tuning, selected using **KDDTrain+ validation Macro F1**. It is **not** claimed to be the best model on KDDTest+.
12. **Model artifact.** `models/nsl_kdd_random_forest_500.joblib` (preprocessor + classifier pipeline).
13. **Deployment readiness.** Serialized pipeline, metadata, input schema, output classes, environment versions; a backend can `joblib.load` and call `predict`. The full-stack app is **not** built in Day 10.
14. **Limitations.** Rare-class recall on KDDTest+ (especially R2L and U2R) is poor; validation overstates generalization; U2R validation support is tiny (n=10); the prediction vector was not serialized inside the Day 5 notebook JSON.

The Random Forest was selected using KDDTrain+ validation Macro F1. KDDTest+ was evaluation-only and did not influence model selection.

## 13. Deployment Readiness

The project now has:

- a locked trained Random Forest (500 trees, validation-selected)
- a serialized preprocessing + classifier `Pipeline`
- metadata JSON
- a documented input schema and column order
- documented output classes (Normal, DoS, Probe, R2L, U2R)
- reproducible environment information (`requirements.txt` plus recorded versions)
- validation smoke-test verification of the packaged artifact

A future backend can load `models/nsl_kdd_random_forest_500.joblib` with `joblib.load` and run inference on records that match the input contract. **No backend is built in Day 10.**

## 14. Final Checklist

In [29]:
print("Expected notebooks:")
for name in expected_nbs:
    print(" ", name, (NB_DIR / name).exists())
print("03_*.ipynb intentionally absent:", not any(NB_DIR.glob("03_*.ipynb")))
art = MODELS_DIR / "nsl_kdd_random_forest_500.joblib"
meta = MODELS_DIR / "nsl_kdd_random_forest_500_metadata.json"
print("Model artifact:", art.exists(), art if art.exists() else "")
print("Metadata artifact:", meta.exists(), meta if meta.exists() else "")
print("README status:", "present" if list(ROOT.glob("README*")) else "absent")
print("requirements.txt:", (ROOT / "requirements.txt").exists())
print(".gitignore:", (ROOT / ".gitignore").exists())
print()
print("Git status:")
print(git("status", "-sb"))
print()
print("Day 10 performs finalization and packaging only. No new model selection or KDDTest+-based tuning is performed.")


Expected notebooks:
  01_eda.ipynb True
  02_baseline_model.ipynb True
  04_smote_experiments.ipynb True
  05_random_forest.ipynb True
  06_model_analysis.ipynb True
  07_final_model_comparison.ipynb True
  08_final_evaluation.ipynb True
  09_reproducibility_audit.ipynb True
  10_finalization_and_packaging.ipynb True
03_*.ipynb intentionally absent: True
Model artifact: True D:\IIITB\network-traffic-anomaly-detection\models\nsl_kdd_random_forest_500.joblib
Metadata artifact: True D:\IIITB\network-traffic-anomaly-detection\models\nsl_kdd_random_forest_500_metadata.json
README status: absent
requirements.txt: True
.gitignore: True

Git status:
## main...origin/main
?? models/
?? notebooks/10_finalization_and_packaging.ipynb

Day 10 performs finalization and packaging only. No new model selection or KDDTest+-based tuning is performed.
